# Streaming

<img src="./assets/LC_streaming.png" width="400">

Streaming reduces the latency between generating data and the user receiving it.
There are two types frequently used with Agents:

## Setup

Load and/or check for needed environmental variables

In [ ]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

In [ ]:
from langchain.agents import create_agent
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI # Ensure this is installed



In [11]:
agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    system_prompt="You are a full-stack comedian",
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


## No Streaming (invoke)

In [12]:
result = agent.invoke({"messages": [{"role": "user", "content": "Tell me a joke"}]})
print(result["messages"][1].content)

Okay, here's one from my comedy stack:

Why did the full-stack developer break up with the database?

...Because he had too many *relationships*, and she just couldn't *commit*!

*(rimshot sound effect generated by a serverless function)*


## values
You have seen this streaming mode in our examples so far. 

In [13]:
# Stream = values
for step in agent.stream(
    {"messages": [{"role": "user", "content": "Tell me a Dad joke"}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me a Dad joke
================================== Ai Message ==================================

Why did the scarecrow win an award?

Because he was outstanding in his field!


## messages
Messages stream data token by token - the lowest latency possible. This is perfect for interactive applications like chatbots.

In [14]:
for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "Write me a family friendly poem."}]},
    stream_mode="messages",
):
    print(f"{token.content}", end="")

The sky is a canvas, so wide and so blue,
With fluffy white clouds, for me and for you.
They float by so gently, a soft, silent stream,
Like pillows of wonder, right out of a dream.

"Look, there's a bunny!" a small voice might say,
"And over there, a dog, ready to play!"
A castle with towers, majestic and grand,
Or maybe a dragon, right over the land.

A ship with white sails, on an ocean of air,
A big sleepy bear, without a care.
They change and they wiggle, they twist and they turn,
New pictures appearing, for all eyes to learn.

So next time you're out, with a smile on your face,
Just peek at the sky, at your very own pace.
For clouds are like stories, that drift and they roam,
And imagination makes them feel right at home!

## Tools can stream too!
Streaming generally means delivering information to the user before the final result is ready. There are many cases where this is useful. A `get_stream_writer` writer allows you to easily stream `custom` data from sources you create.

In [17]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()
    # stream any arbitrary data
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"


agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[get_weather],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "custom"],
):
    print(chunk)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='70570db1-b663-47d9-8e72-45e03fa506dc')]})
('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='70570db1-b663-47d9-8e72-45e03fa506dc'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "San Francisco"}'}, '__gemini_function_call_thought_signatures__': {'4cf326f5-da46-42d7-815a-dccb74b0660b': 'CvQBARFNMg/PeBYnNXOUF2zB9qyG4j4Y52QzyDAnFAo0x6HlZ+ZhARLYysOYBfI/v3yCGiAqQwBKJJ7Z9W2TLiVHzm1PnfK1yG0xLofhC7ymFMZ2sWg1gG0mN9QETAinJieXPE0653M2HnUhiJe3norhIvZoyBxJYR9kXkUzR4AS4OTHZF/YPa6Nnw3wY5f3cACBR/PDTWRB0Q2nU3omQNA35PQ260fxG+89yYngVU53clyDrn9nAihRy/YVJ3lwpmEYChHDCq3l0+XKYSS44cm3zWFEKj74YE1lQZG0pahbn1IwG0L5PwMFjaS3FbmzavmNL9qCWQ=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': '

In [18]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["custom"],
):
    print(chunk)

('custom', 'Looking up data for city: San Francisco')
('custom', 'Acquired data for city: San Francisco')


## Try different modes on your own!
Modify the stream mode and the select to produce different results.

In [23]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "messages", "custom"],
):
    if chunk[0] == "messages":
        print(chunk[1])

(AIMessageChunk(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "San Francisco"}'}, '__gemini_function_call_thought_signatures__': {'b05ae3cc-9de2-494a-b74f-4fce33c8e7aa': 'CiQBEU0yD5XWzxK9WSHf0dbB9KVankiLX++aky8zOTkfSIzU/OUKWgERTTIPnfU3vE6Jbl6v9RvVNcZNMOocPmz/Pu/P7uR70sue54Sji38NwHq+YD4zIHgSu4autxBNsPON5v3Sm72r4EsW0UcupIwFre4rASpFftQUlO4xnOfmSgq5AQERTTIPo5o2n54R5Vff642FE2ulk5+cLQbcs7qIGfTYWW56cRtmQsS2/Ccb4oA/kUD0iM2jvx4ERaS3fPPHnBrMdOzefO3QBGxLHk5Z9t/Mlq9IZX1vPrvmSsoJOD2MMKcsP5iqPvCVn/wkFzAg4vl5hc1zQFxL+6KrJjS8Z+CKIk7Ykl7zIOS5WF/5TlS9hFtvxJ2u/BckqldbWIK8M6n5Idd7kraVcISjxXDVpDk6MkgHG0ohBQwh'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f6cae-9fcf-7f83-803b-819189d1e08e', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 'b05ae3cc-9de2-494a-b74f-4fce33c8e7aa', 'type': 'tool_call'}], invalid_tool_calls=[], us